# Groundhog — notebook demo
A lightweight, **real** experiment loop: we train a small softmax classifier across a few configs and watch Groundhog memory fill up — Pre-flight Guard, semantic recall, and derived insights — all scoped to one project.

Prereqs: the 3 services running (cognee :8010, backend :8000, frontend :5173).

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd())=='pages' else os.getcwd())
# make sure repo root is importable so `demo` package + sdk resolve
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, ROOT)
from demo._bootstrap import init, BACKEND
from demo.train_lib import train_and_evaluate, DATASET_INFO
import groundhog

project_id = init(experiment='blobs3_notebook')
print('connected to project:', project_id)

### Register the Groundhog notebook magics
`%groundhog_hypothesis` and `%groundhog_decision` write the *why* straight into project memory.

In [ ]:
get_ipython().run_line_magic('load_ext', 'connectors.jupyter_magic')

### Dataset (proper, versioned input)

In [ ]:
DATASET_INFO

### Hypothesis → memory

In [ ]:
%groundhog_hypothesis A little weight decay should improve val_accuracy on blobs3 without hurting convergence.

### Run a small weight-decay sweep
`groundhog.run(...)` runs the Pre-flight Guard on enter and auto-logs metrics + wall-clock on exit. Re-running this cell should show the guard flag configs you already tried.

In [ ]:
EXPERIMENT = 'blobs3_notebook'
for wd in (0.0, 1e-3, 1e-2):
    cfg = {'model':'softmax','optimizer':'sgd','lr':0.1,'batch_size':32,'weight_decay':wd,'epochs':20}
    with groundhog.run(config=cfg, experiment=EXPERIMENT) as r:
        if r.already_tried:
            print(f'wd={wd}: already tried -> {r.prior_metrics}; skipping'); r.skip()
        else:
            out = os.path.join('outputs', EXPERIMENT, f'wd_{wd}')
            m = train_and_evaluate(cfg, out_dir=out)
            r.log(**m)
            print(f'wd={wd}: val_acc={m["val_accuracy"]} val_loss={m["val_loss"]}')

### Record a decision

In [ ]:
%groundhog_decision Keeping weight_decay small (<=1e-3); 1e-2 underfits blobs3.

### Ask the memory

In [ ]:
print(groundhog.query('Across the notebook runs, which weight_decay gave the best val_accuracy and why?'))

### What the memory learned (derived insights)

In [ ]:
import json, urllib.request
url = f'{BACKEND}/api/insights?project_id={project_id}'
ins = json.load(urllib.request.urlopen(url, timeout=60))
print('summary:', ins.get('summary'))
for s in ins.get('parameter_sensitivity', [])[:5]:
    print(f"  {s['parameter']}: impact={s['sensitivity']} best={s['best_value']}")

Open the dashboard at **http://localhost:5173** and select this project — the runs, artifacts, and insights you just created are all there.